# Step 2 (Approach 2): Train Simple Unsupervised VAE & Run FAISS Candidate Blocking

This notebook trains the **Simple Unsupervised Variational Autoencoder (Simple VAE)** model on SageMaker GPU, extracts 256-dimensional normalized latent vectors $z$ for all test entities, runs country-partitioned FAISS blocking, and exports `candidate_pairs.tsv`.

In [ ]:
# 1. Install required packages directly into active SageMaker kernel
%pip install -q torch transformers sentence-transformers faiss-cpu catboost boto3 tqdm unidecode indic-transliteration

import os
import sys
import pandas as pd
import torch

# Ensure student_resource, approach_2_simpleVAE, and current working directories are on sys.path
current_dir = os.path.dirname(os.path.abspath("")) if os.path.abspath("") else os.getcwd()
student_resource_dir = os.path.dirname(current_dir) if ("approach_" in os.path.basename(current_dir) or "global_" in os.path.basename(current_dir)) else current_dir

for p in [student_resource_dir, current_dir]:
    if p and p not in sys.path:
        sys.path.insert(0, p)

try:
    from approach_2_simpleVAE.config import path_config, model_config
    from approach_2_simpleVAE.src.trainer import train_simple_vae
    from approach_2_simpleVAE.src.blocking import run_blocking_pipeline
    from approach_2_simpleVAE.src.s3_utils import upload_file_to_s3
except ImportError:
    from config import path_config, model_config
    from src.trainer import train_simple_vae
    from src.blocking import run_blocking_pipeline
    from src.s3_utils import upload_file_to_s3

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 1. Train Unsupervised Simple VAE Model on GPU
Trains the transformer backbone + ELBO reconstruction loss over unsupervised record text strings.

In [ ]:
print(f"Training Simple VAE for {model_config.epochs} epochs on device: {model_config.device}...")

checkpoint_file = train_simple_vae(
    epochs=model_config.epochs,
    batch_size=model_config.batch_size,
    learning_rate=model_config.learning_rate,
    save_s3=True
)

print(f"--> Saved best Simple VAE checkpoint to: {checkpoint_file}")

## 2. Encode Test Set & Run FAISS Candidate Blocking
Passes test entities through Simple VAE, queries country-partitioned FAISS vector indices, and writes `candidate_pairs.tsv`.

In [ ]:
print("Loading test datasets...")
test_s1 = pd.read_csv(os.path.join(path_config.test_dir, "test_source1.tsv"), sep="\t")
test_s2 = pd.read_csv(os.path.join(path_config.test_dir, "test_source2.tsv"), sep="\t")
test_s3 = pd.read_csv(os.path.join(path_config.test_dir, "test_source3.tsv"), sep="\t")

candidate_tsv_path, candidate_map = run_blocking_pipeline(
    checkpoint_path=checkpoint_file,
    s1_df=test_s1,
    s2_df=test_s2,
    s3_df=test_s3,
    output_candidate_path=os.path.join(path_config.output_dir, "candidate_pairs.tsv")
)

# Upload candidate_pairs.tsv to S3
s3_key = f"{path_config.s3_prefix}/output/candidate_pairs.tsv"
upload_file_to_s3(candidate_tsv_path, path_config.s3_bucket, s3_key)

print(f"\n--> STEP 2 COMPLETE: candidate_pairs.tsv exported successfully for {len(test_s1):,} S1 entities!")
print("Next step: Open global_notebooks/03_global_feature_extraction_and_reranking.ipynb to score and format final submission!")